Setup - copied from master testing

In [ ]:
import requests
import os
from dotenv import load_dotenv
from requests.auth import HTTPBasicAuth
import time

validationEnabled = False

load_dotenv()

v2Server = os.getenv("V2_SERVER")
fhirServer = os.getenv("FHIR_SERVER")
toolsServer = os.getenv("V2_TOOLS")

tokenUrl = os.getenv("OAUTH2_TOKEN")
clientId = os.getenv("CLIENT_ID")
clientSecret = os.getenv("CLIENT_SECRET")

the_data = {"grant_type": "client_credentials", "scope": "system/*.*"}
headers = {'Content-Type': 'application/x-www-form-urlencoded'}
response = requests.post(tokenUrl, auth=HTTPBasicAuth(clientId, clientSecret),verify=False,data=the_data, headers=headers)
responseInclude = response.json()
token = responseInclude['access_token']

print(token)
print("token was displayed")

In [ ]:
import fhirclient.models.binary as bin

headersFHIR = {'Content-Type': 'application/fhir+json',
               'Authorization': 'Bearer ' + token}
headersV2 = {'Content-Type': 'x-application/hl7-v2+er7'}

In [ ]:
import base64
import json

test_report_list = ['cepheid-1.txt','cepheid-2.txt','cepheid-3.txt','cepheid-4.txt','cepheid-5.txt','cepheid-6.txt','cepheid-7.txt']

for file in test_report_list:
    with open('Input/V2/R32/' + file, 'rb') as g:
        s = g.read()
        print(file)
        rFHIR = requests.post(toolsServer + '/transformToFHIR', data = s,verify=False,headers=headersV2)
        ##print(r.status_code)
        with open('Output/FHIR/R32/' + file + '.json', 'w',encoding='utf-8', errors='replace') as vFHIR:
            vFHIR.write(rFHIR.text)
            vFHIR.close()

        ##rV2 = requests.post(toolsServer + '/transformToV2', data = rFHIR.text,verify=False,headers=headersFHIR)

        ##with open('Output/V2/R32/' + file, 'w',encoding='utf-8', errors='replace') as v2:
        ##    v2.write(rV2.text)
        ##    v2.close()

        r = requests.post(v2Server, data = s)
        print(r.status_code)
        print(r.text)

        response1JSON = rFHIR.json()
        for entry in response1JSON['entry']:
            resource = entry['resource']
            if resource['resourceType'] == 'MessageHeader':
                resource['eventCoding']['code'] = 'T02'
                entry['resource'] = resource
            if resource['resourceType'] == 'Binary':
                try:
                    binary = bin.Binary(resource)
                    encoded = binary.data
                    decode = base64.b64decode(encoded)
                    with open('Output/PDF/R32/' + file+ '.pdf', 'wb') as pdf:
                        pdf.write(decode)
                except ValueError:
                    print('Exception in '+file)
                    # This should be wales only

        jsonStr = json.dumps(response1JSON, indent=4)
